# 实验三：QQ邮箱采集（Playwright版）

班级：数据1241　姓名：沈昊辉　学号：112404260123

四阶段写在同一个 Notebook 中，按单元 1—13 顺序补全：

| 阶段 | 单元 | 完成效果 |
|---|---|---|
| 1 扫码登录 | 1—2 | 未扫码时等待；扫码后输出“登录成功”，进入收件箱 |
| 2 采集和保存 | 3—6 | 采集 10 封普通邮件，pandas 保存并回读核对 |
| 3 滚动采集 | 7—9 | 连续采集 3 批，按邮件 ID 去重 |
| 4 详情正文 | 10—12 | 逐封读取正文并返回列表 |

运行顺序：单元1 → 2 →（人工扫码）→ 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 收尾（脱敏导CSV）→ 单元14。单元13 是附件下载选做，不选做就不运行。

单元14 不是指导书要求，是自建的收尾单元：把本次运行的实测值（条数、批次日志、ID 唯一性、正文长度、附件大小）自动写进 `实验报告-Playwright版.md` 里的 `[[xxx]]` 标记，省掉手工逐格填。

目录约定：`private_data/` 放原始 CSV 和附件（只在本地），`data/` 放脱敏 CSV（提交用），`images/` 放截图。原始邮件、私人正文和登录状态都不上传。

## 第一阶段：扫码登录（第1—2课时）

### 单元1：准备浏览器

`start()` 启动工具，`launch()` 启动浏览器，`new_context()` 创建环境，`new_page()` 创建页面。可视化运行需要 `headless=False`，显示服务由 `course_visual.prepare()` 提供。

In [1]:
import sys
from pathlib import Path
from uuid import uuid4
from importlib.metadata import version as pkg_version

import pandas as pd
from playwright.async_api import async_playwright
from playwright.async_api import TimeoutError as PlaywrightTimeoutError

# 原始数据与脱敏数据分开放：private_data 只在本地，data 用于提交。
PRIVATE = Path.cwd() / "private_data"
PUBLIC = Path.cwd() / "data"
IMAGES = Path.cwd() / "images"
for folder in (PRIVATE, PUBLIC, IMAGES):
    folder.mkdir(exist_ok=True)

# 报告自动填充用：各阶段把实测值记进这个字典，最后一个单元统一写进报告。
STATS = {}

# 使用visual安装时已有的显示服务配置（容器内的Xvfb显示）。
visual_dir = Path.cwd().parent / "L07_L08_visual"
if not (visual_dir / "course_visual.py").is_file():
    raise RuntimeError(
        "未找到 workspace/L07_L08_visual：请先双击 install-visual.bat 安装增量包，"
        "再运行 start-course-visual.bat 启动显示服务。"
    )
sys.path.insert(0, str(visual_dir))
import course_visual
course_visual.prepare()

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, args=["--disable-dev-shm-usage"])
context = await browser.new_context(
    viewport={"width": 1000, "height": 620}, accept_downloads=True
)
page = await context.new_page()
page.set_default_timeout(20000)  # 配置默认超时，单位毫秒，不需要await。

print("Python     :", sys.version.split()[0])
print("Playwright :", pkg_version("playwright"))
print("pandas     :", pd.__version__)

Python     : 3.12.3
Playwright : 1.62.0
pandas     : 3.0.5


### 单元2：打开邮箱并等待扫码

`page.goto()` 打开网页；定位器只描述位置，`wait_for()` 才真正等待。等待上限取 600000 毫秒（10 分钟），界面一出现就往下走，不用固定 `sleep`。

QQ 邮箱的邮件界面**可能套在 iframe 里**，所以这一步不只看顶层文档，而是把所有 frame 逐个看一遍，谁出现邮件行或"收件箱"就认谁，结果存进 `MAIL`。后面所有邮件相关的定位都在 `MAIL` 上做——没有 iframe 时 `MAIL` 就是整个页面，写法不用改。

> 完成证据：登录代码、`登录成功` 输出和收件箱截图；截图里框出定位、等待、点击三行，并遮挡账号等私人信息。

In [2]:
# ① 打开邮箱首页。domcontentloaded 只等DOM就绪，不等图片等静态资源。
await page.goto("https://mail.qq.com/", wait_until="domcontentloaded")
print("请在浏览器画面中扫码", flush=True)


# ② 跨frame找邮件界面：先看谁的页面里有邮件行（最可靠），没有再退一步找“收件箱”文字。
async def wait_login(max_rounds=400, interval_ms=1500):
    # 400 轮 × 1.5 秒 ≈ 10 分钟。用轮数计上限，不依赖 time 模块。
    for _ in range(max_rounds):
        for frame in page.frames:
            try:
                if await frame.locator("[data-mailid]").count():
                    return frame
            except Exception:
                continue  # frame 正在跳转时读不到，跳过它，下一轮再看
        for frame in page.frames:
            try:
                if await frame.locator("body").get_by_text("收件箱", exact=True).count():
                    return frame
            except Exception:
                continue
        await page.wait_for_timeout(interval_ms)
    return None


# ③ 最长等待10分钟。邮件界面出现即视为登录成功，立即继续。
MAIL = await wait_login()
if MAIL is None:
    raise RuntimeError(
        "等了10分钟仍没进到邮件列表：请在浏览器画面里扫码，或确认画面已经是收件箱后重跑本单元。"
    )
STATS["frame_url"] = MAIL.url
print("登录成功（邮件界面在：", "顶层文档" if MAIL.parent_frame is None else MAIL.url[:70], "）", flush=True)

# ④ 点一下“收件箱”，让列表回到第一页。已经在收件箱时，这一步只是确认。
inbox = MAIL.locator("body").get_by_text("收件箱", exact=True).first
if await inbox.count():
    await inbox.click()
    print("已点击收件箱", flush=True)
else:
    print("没找到“收件箱”文字，但邮件行已出现，直接继续", flush=True)

请在浏览器画面中扫码
登录成功（邮件界面在： 顶层文档 ）
已点击收件箱


## 第二阶段：采集10封邮件并保存（第3—4课时）

### 单元3：自己找选择器

两条线索：发件人是 `.cmp-account-nick`，广告聚合条目带 `.gg-unread-count`（必须跳过）。其余在开发者工具（F12 → Elements）里找：一封邮件的外层节点、主题、列表时间、发件人悬停区域、悬停卡片里的邮箱。

每个字段留两组候选，登录后用第一行的真实 DOM 自检一次，取真正匹配到的那组——这样换账号、换模板时不至于整段失效。自检结果会打印出每个字段命中的数量与首行取值，**计数为 0 就把 F12 里看到的类名加进对应候选列表再重跑本单元**。

In [3]:
# —— 邮件行：每封邮件的外层节点带 data-mailid 属性，用它作为一封邮件的唯一标识 ——
ROW_CANDIDATES = [".mail-list-page [data-mailid]", "[data-mailid]"]

# —— 发件人昵称（线索给出）与广告聚合入口标记（应跳过） ——
SENDER_CANDIDATES = [".cmp-account-nick", "[class*='account-nick']"]
AD_CSS = ".gg-unread-count"

# —— 主题、列表时间、悬停区域、卡片中的邮箱 ——
SUBJECT_CANDIDATES = ["[class*='mail-subject']", "[class*='subject']"]
TIME_CANDIDATES = ["[class*='mail-time']", "[class*='time']"]
HOVER_CANDIDATES = [".cmp-account-nick", "[class*='account-nick']"]
EMAIL_CANDIDATES = ["[class*='account-email']", "[class*='mail-address']"]


async def pick_css(candidates, scope=None):
    """按候选顺序返回第一个在本页/本行真正匹配到节点的写法。"""
    for css in candidates:
        if await (scope or MAIL).locator(css).count():
            return css
    return candidates[0]


ROW_CSS = await pick_css(ROW_CANDIDATES)
SENDER_CSS = await pick_css(SENDER_CANDIDATES)
SUBJECT_CSS = await pick_css(SUBJECT_CANDIDATES)
TIME_CSS = await pick_css(TIME_CANDIDATES)
HOVER_CSS = await pick_css(HOVER_CANDIDATES)
EMAIL_CSS = await pick_css(EMAIL_CANDIDATES)

# —— 自检：各字段在首行邮件里命中几个节点，取到的是什么文字 ——
await MAIL.locator(ROW_CSS).first.wait_for(timeout=60000)
sample_row = MAIL.locator(ROW_CSS).first
# 收件箱列表当前渲染出多少行邮件，记下来给报告用。
STATS["inbox_rows"] = await MAIL.locator(ROW_CSS).count()
print("邮件行数量 :", STATS["inbox_rows"], "  ", ROW_CSS)
for label, css in (("发件人", SENDER_CSS), ("主题", SUBJECT_CSS), ("时间", TIME_CSS)):
    count = await sample_row.locator(css).count()
    text = (await sample_row.locator(css).first.inner_text())[:28] if count else "—"
    print(f"{label:<4}: {count:>2} 个  {css:<28} 首行取值：{text}")
print("邮箱卡片需悬停后才进DOM，留到单元5第一条邮件时核对：", EMAIL_CSS)

邮件行数量 : 24    .mail-list-page [data-mailid]
发件人 :  1 个  .cmp-account-nick            首行取值：QQ音乐项目组
主题  :  1 个  [class*='mail-subject']      首行取值：恭喜您，成功开通豪华绿钻
时间  :  1 个  [class*='mail-time']         首行取值：11/23
邮箱卡片需悬停后才进DOM，留到单元5第一条邮件时核对： [class*='account-email']


### 单元4：定义列表解析函数

`evaluate_all()` 对整批元素执行一次 JavaScript，一次拿全部 ID，避免边更新边取旧元素。悬停取邮箱分四步：鼠标先停到列表页顶部空白 → 等上一张卡片消失 → 把本行发件人滚进可见区域并等滚动状态类消失 → 再悬停（被挡就 Esc 重试，两轮不行用 `force=True` 兜底）、等卡片可见。

`get_row()` 每次都用 ID 重新定位，不保存元素对象，列表刷新后旧引用也不会失效。悬停发件人可能被残留浮层挡住（Playwright 报 `<div> intercepts pointer events`），所以悬停分三级：常规悬停 → 按 Esc 关浮层、鼠标停到列表空白处再试 → `force=True` 跳过遮挡检查兜底。

In [4]:
async def get_ids():
    await MAIL.locator(ROW_CSS).first.wait_for()
    # 一次读取整批ID，避免列表更新时逐个保存旧元素。
    ids = await MAIL.locator(ROW_CSS).evaluate_all(
        "rows => rows.map(row => row.getAttribute('data-mailid'))"
    )
    # 列表更新瞬间可能有个别节点还没挂上ID，去空、去重后再校验。
    ids = [mail_id for mail_id in dict.fromkeys(ids) if mail_id]
    if not ids:
        raise ValueError("没有取得有效邮件ID，请检查页面")
    return ids


def get_row(mail_id):
    return MAIL.locator(f'{ROW_CSS}[data-mailid="{mail_id}"]').first


async def hide_hover_card():
    """等所有候选浮层都消失（最多等3秒），确保接下来读到的是本行卡片，不是上一封的。"""
    for css in EMAIL_CANDIDATES:
        try:
            await MAIL.locator(css).first.wait_for(state="hidden", timeout=3000)
        except PlaywrightTimeoutError:
            pass  # 没消失也继续：稍后会重新悬停，读到的必须“可见”才算数


async def read_hover_email():
    """悬停后卡片是异步渲染的，hover 返回的瞬间还没进 DOM，所以必须“等出现”而不是当场查数量。

    候选逐个试：等它可见（最多4秒），可见且非空才算命中。
    """
    global EMAIL_CSS
    for css in EMAIL_CANDIDATES:
        node = MAIL.locator(css).first
        try:
            await node.wait_for(state="visible", timeout=4000)
        except PlaywrightTimeoutError:
            continue
        text = (await node.inner_text()).strip()
        if text:
            EMAIL_CSS = css
            return text
    print("警告：没有读到邮箱，请检查 EMAIL_CANDIDATES")
    return ""


async def park_mouse(mail_id=None):
    """把鼠标移开悬停目标。优先停到本行时间列（鼠标不出列表，不触发工具栏提示层，
    这是昨晚 62/62 全读到的稳妥做法）；没有行上下文时才退回列表页顶部空白。"""
    if mail_id is not None:
        try:
            await get_row(mail_id).locator(TIME_CSS).first.hover(timeout=5000)
            return
        except PlaywrightTimeoutError:
            pass
    try:
        await MAIL.locator(".mail-list-page").first.hover(
            position={"x": 420, "y": 8}, timeout=3000, force=True)
    except PlaywrightTimeoutError:
        await page.mouse.move(420, 8)


async def hover_sender(mail_id):
    """悬停发件人。偶发被残留浮层挡住导致 hover 超时，三级退让：
    常规悬停 → 按 Esc 关浮层、鼠标停到本行时间列重试 → force=True 跳过遮挡检查兜底。"""
    target = get_row(mail_id).locator(HOVER_CSS).first
    try:
        await target.hover(timeout=8000)
        return
    except PlaywrightTimeoutError:
        pass
    # 常规悬停被挡：先清浮层再试一轮。
    await page.keyboard.press("Escape")
    await park_mouse(mail_id)
    await hide_hover_card()
    try:
        await target.hover(timeout=8000)
        return
    except PlaywrightTimeoutError:
        pass
    # 仍被挡：force=True 跳过遮挡检查。卡片可能打不开，读不到就按空邮箱处理。
    await target.hover(timeout=8000, force=True)


async def read_list_mail(mail_id):
    row = get_row(mail_id)
    # 广告聚合入口不是普通邮件，跳过。
    if await row.locator(AD_CSS).count():
        return None
    sender = (await row.locator(SENDER_CSS).first.inner_text()).strip()
    subject = (await row.locator(SUBJECT_CSS).first.inner_text()).strip()
    time_text = (await row.locator(TIME_CSS).first.inner_text()).strip()

    # 先把鼠标停到本行时间列，等上一张卡片消失，避免残留浮层挡住指针。
    await park_mouse(mail_id)
    await hide_hover_card()
    # 先把发件人滚入可见区域。滚动时页面会暂时停止响应悬停。
    await row.locator(HOVER_CSS).first.scroll_into_view_if_needed()
    await MAIL.locator(".mail-list-page .ui-float-scroll-body-scrolling").first.wait_for(state="hidden")
    # 滚动结束后，再悬停本封发件人，等卡片显示后读取邮箱。
    await hover_sender(mail_id)
    email = await read_hover_email()
    if not email:
        # 连续采集时个别邮件的卡片偶尔不渲染（上一封的清理和本封的渲染撞车）：
        # 清浮层后完整重试一次，还读不到就打印"到底被什么挡住"再按空邮箱处理。
        print(f"  {mail_id[:20]} 邮箱没读到，清浮层重试一次", flush=True)
        await page.keyboard.press("Escape")
        await park_mouse(mail_id)
        await hide_hover_card()
        await row.locator(HOVER_CSS).first.scroll_into_view_if_needed()
        await page.wait_for_timeout(800)
        await hover_sender(mail_id)
        email = await read_hover_email()
        if not email:
            blocker = await row.locator(HOVER_CSS).first.evaluate(
                """el => {
                    const r = el.getBoundingClientRect();
                    const hit = document.elementFromPoint(r.x + r.width / 2, r.y + r.height / 2);
                    if (!hit) return 'elementFromPoint 无命中';
                    return hit.tagName + '.' + String(hit.className).slice(0, 50);
                }""")
            print(f"  诊断：发件人位置被 {blocker} 挡住", flush=True)
    return {"邮件ID": mail_id, "发件人": sender,
            "邮箱": email, "主题": subject, "时间": time_text}


print("已定义 5 个函数：get_ids / get_row / hide_hover_card / read_hover_email / read_list_mail")

已定义 5 个函数：get_ids / get_row / hide_hover_card / read_hover_email / read_list_mail


### 单元5：采集10封

按页面顺序逐封解析，广告条目返回 `None` 被跳过，凑满 10 封就停。

In [5]:
basic_records = []
for mail_id in await get_ids():
    record = await read_list_mail(mail_id)
    if record is None:
        continue
    basic_records.append(record)
    if len(basic_records) == 10:
        break
print("成功采集：", len(basic_records))

成功采集： 10


### 单元6：用pandas保存和回读

`DataFrame()` 建表、`to_csv()` 保存、`read_csv()` 回读；`index=False` 不写行号，`utf-8-sig` 让 Excel 直接打开不乱码。这两个函数后面阶段继续用，pandas 调用不加 `await`。

> 完成证据：字段提取与 pandas 代码截图；10 条记录的回读结果；第一、中间、最后一封与网页一致的核对截图（脱敏）。

In [6]:
def save_records(records, filename, columns):
    df = pd.DataFrame(records, columns=columns)
    path = PRIVATE / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def read_records(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


BASIC_COLUMNS = ["发件人", "邮箱", "主题", "时间"]
basic_path = save_records(basic_records, "mail_basic.csv", BASIC_COLUMNS)
basic_df = read_records(basic_path)
print("CSV记录数：", len(basic_df))
basic_df.head(10)

CSV记录数： 10


,发件人,邮箱,主题,时间
0,QQ音乐项目组,music@tencent.com,恭喜您，成功开通豪华绿钻,11/23
1,Microsoft Rewards,MicrosoftRewards@customeremail.microsoftreward...,奖品包括 Sonos 音箱、Apple AirPods Pro、礼品卡等,11/10
2,Microsoft Rewards,MicrosoftRewards@customeremail.microsoftreward...,免费获取 Super Duolingo,10/28
3,GitHub,noreply@github.com,[GitHub] A first-party GitHub OAuth applicatio...,10/27
4,赛氪,service@email.saikr.com,【开考通知】本周开考！预计赛后三天可领取三学会盖章证书！,10/27
5,MathWorks Trial Team,mathworks@go.mathworks.com,您的试用即将结束,10/20
6,Microsoft Rewards,MicrosoftRewards@customeremail.microsoftreward...,(广告)探索今天的 XBOX 赠品,9/28
7,MathWorks Trial Team,mathworks@go.mathworks.com,试用下一步–立即访问资源,9/22
8,service,service@mail.account.mathworks.com,验证电子邮件地址,9/22
9,Docker,no-reply@notify.docker.com,[Docker] You + Docker = Ready for Action,9/18


## 第三阶段：滚动采集与ID去重（第5—6课时）

### 单元7：补全解析和批次循环

滚动用的三个属性：`scrollTop` 当前位置、`clientHeight` 一屏可见高度、`scrollHeight` 内容总高度。下一屏是 `top + height`，末尾最大位置是 `scrollHeight - clientHeight`，两者取小。

滚动后不等固定时间，而是用 `wait_for_function()` 在浏览器里比较新旧 ID 列表，真正变化了才继续——这比 `sleep` 稳，也不会白等。

> 完成证据：滚动、等待 ID 变化、停止条件的代码与真实批次日志。

In [7]:
SCROLL_CANDIDATES = [".mail-list-page .ui-float-scroll-body", ".mail-list-page [class*='scroll-body']"]
SCROLL_CSS = await pick_css(SCROLL_CANDIDATES)


def get_box():
    return MAIL.locator(SCROLL_CSS).first


print("滚动容器：", SCROLL_CSS,
      " 可见高度", await get_box().evaluate("box => box.clientHeight"),
      " 总高度", await get_box().evaluate("box => box.scrollHeight"))


async def parse_batch(mail_ids, seen_ids, include_details=False, max_mails=0):
    result = []
    for mail_id in mail_ids:
        if max_mails and len(seen_ids) >= max_mails:
            break
        if mail_id in seen_ids:
            continue
        record = await read_list_mail(mail_id)
        if record is None:
            continue
        if include_details:
            # 第四阶段定义正文函数；本阶段False，不执行这里。
            record.update(await read_detail(mail_id))
        result.append(record)
        seen_ids.add(mail_id)
    return result


async def collect_batches(include_details=False, max_mails=0):
    await inbox.click()
    await MAIL.locator(ROW_CSS).first.wait_for()
    await get_box().evaluate("box => box.scrollTop = 0")
    records, seen_ids = [], set()
    batches, stop_reason = [], "完成3批"
    for batch in range(3):
        top = await get_box().evaluate("box => box.scrollTop")
        result = await parse_batch(await get_ids(), seen_ids, include_details, max_mails)
        records.extend(result)
        batches.append((len(result), len(seen_ids)))
        print(f"第{batch + 1}批，新增{len(result)}，累计{len(seen_ids)}")
        if max_mails and len(seen_ids) >= max_mails:
            stop_reason = f"达到 max_mails={max_mails} 的上限（试采用）"
            break
        if batch == 2:
            break
        box = get_box()
        height = await box.evaluate("box => box.clientHeight")
        full_height = await box.evaluate("box => box.scrollHeight")
        next_top = min(top + height, full_height - height)
        if next_top <= top:
            stop_reason = "已到列表末尾（next_top <= top）"
            print("已到列表末尾")
            break
        old_ids = await get_ids()
        await box.evaluate("(box, value) => box.scrollTop = value", next_top)
        # 在浏览器中比较新旧ID，改变后才继续；万一没变化也不让整批中断。
        try:
            await MAIL.wait_for_function("""data => {
                const ids = Array.from(document.querySelectorAll(data.selector),
                                       node => node.getAttribute('data-mailid'));
                return ids.length > 0 && JSON.stringify(ids) !== JSON.stringify(data.ids);
            }""", arg={"selector": ROW_CSS, "ids": old_ids}, timeout=20000)
        except PlaywrightTimeoutError:
            print("提示：列表ID未在20秒内变化，继续下一批（已采ID会被去重集合跳过）")
    if not include_details:
        # 只有第三阶段的正式三批要写进报告的批次表，后面的详调试采不覆盖它。
        STATS["page_batches"] = batches
        STATS["page_stop"] = stop_reason
    return records, seen_ids

滚动容器： .mail-list-page .ui-float-scroll-body  可见高度 512  总高度 2442


### 单元8：执行并保存3批

三批共用一个去重集合 `seen_ids`；记录数应当等于唯一 ID 数。

In [8]:
PAGE_COLUMNS = ["邮件ID"] + BASIC_COLUMNS
page_records, seen_ids = await collect_batches()
page_path = save_records(page_records, "mail_pages.csv", PAGE_COLUMNS)
page_df = read_records(page_path)
print("CSV记录数：", len(page_df))
print("唯一ID数：", page_df["邮件ID"].nunique())
print("重复ID数：", page_df["邮件ID"].duplicated().sum())

第1批，新增24，累计24
第2批，新增18，累计42
第3批，新增18，累计60
CSV记录数： 60
唯一ID数： 60
重复ID数： 0


### 单元9：重复采集验证

取已采过的 5 个 ID 再解析一遍，**仍用同一个 `seen_ids`**（不是重启程序后覆盖文件）。新增应为 0，CSV 字节不变。

> 完成证据：回读数量、ID 唯一性核对，以及重复采集新增 0 条的结果截图。

In [9]:
replay_ids = [r["邮件ID"] for r in page_records[:5]]
assert replay_ids, "先成功采集一些邮件，再做重复验证"
before = page_path.read_bytes()
replayed = await parse_batch(replay_ids, seen_ids)
print("重复采集新增：", len(replayed))
save_records(page_records + replayed, "mail_pages.csv", PAGE_COLUMNS)
STATS["replay_added"] = len(replayed)
STATS["csv_same"] = before == page_path.read_bytes()
print("CSV内容未变：", STATS["csv_same"])

重复采集新增： 0
CSV内容未变： True


## 第四阶段：邮件详情与正文采集（第7—8课时）

### 单元10：补全正文函数

点主题进详情，读正文，再点“返回”回列表并恢复原来的滚动位置。`finally` 保证即使读正文出错也会退回列表，不会把后面几封卡在详情页。

正文藏在闭包 Shadow DOM 里：读信层的宿主容器（`.mail-detail-content`）在普通 DOM 里永远是空的，`querySelector` 和 XPath 都查不到——只有 Playwright 的选择器能穿透 shadow 边界，所以候选第一位是 shadow 内部的真实正文节点 `.xmail-html-render-body`，后面留几个常规候选兜底。点开邮件后先等详情层渲染再读；三种方式都读不到就打印诊断输出。纯图片邮件没有文字，会如实留空。

In [10]:
# 正文在闭包 Shadow DOM 里：宿主 div.detail-content-box 在普通 DOM 永远是空的，
# querySelector/XPath 查不到，只有 Playwright 选择器能穿透 shadow 读到真实正文节点。
# 所以候选第一个必须放 .xmail-html-render-body（shadow 内部的正文渲染节点）。
BODY_CANDIDATES = [".xmail-html-render-body", "[class*='mail-body']", "[class*='mail-content']", "#mailContentContainer"]
BODY_CSS = BODY_CANDIDATES[0]

BACK_TEXT = "返回列表"
# 注意：不能把通用的“返回”两个字放进候选。实跑发现邮箱侧栏常驻一个设置用的
# “返回”按钮，文字恰好就叫“返回”，点它会把页面带到设置中心，后面全找不到。
BACK_TEXTS = [BACK_TEXT, "返回邮件列表", "回到列表", "返回收件箱"]
# 文字都点不到时再试属性：返回通常是图标按钮，靠 title/aria-label/class 认出来。
# 注意：宽泛的 [class*='back'] 不能要——实跑证明设置侧栏的 back-btn-text 也含
# "back"，点它会把页面带到设置中心。icon-back / btn-back 是读信页图标的形态。
BACK_ATTR_CANDIDATES = [
    "[title='返回列表']", "[aria-label='返回列表']",
    "[title*='返回列表']", "[aria-label*='返回列表']",
    "[class*='icon-back']", "[class*='btn-back']", "[class*='return']",
]
BACK_HINT = None  # 记住上次成功的方式（如 ("text", "返回列表")），下一封直接用它，不用全扫。
DOWNLOAD_ATTACHMENTS = False


async def _first_visible(locator, max_probe=10):
    """在前几个匹配里找可见节点。只查 first 会被前面的隐藏节点挡住。"""
    try:
        total = await locator.count()
    except Exception:
        return None
    for index in range(min(total, max_probe)):
        node = locator.nth(index)
        try:
            if await node.is_visible():
                return node
        except Exception:
            pass
    return None


async def find_in_frames(make_locator):
    """跨所有 frame 找控件。详情页可能让 MAIL 指向的内容变化，不能只查 MAIL。"""
    for frame in page.frames:
        try:
            node = await make_locator(frame)
        except Exception:
            continue  # frame 正在跳转时读不到，换下一个
        if node is not None:
            return node
    return None


async def locate_back():
    """按“上次成功的 → 文字 → 图标属性”的顺序找返回控件，返回（节点, 描述）或 (None, "")。"""
    global BACK_HINT
    if BACK_HINT:
        kind, key = BACK_HINT
        if kind == "text":
            node = await find_in_frames(lambda f, k=key: _first_visible(f.get_by_text(k, exact=True).first))
        else:
            node = await find_in_frames(lambda f, k=key: _first_visible(f.locator(k).first))
        if node is not None:
            return node, f"{kind}“{key}”"
    for text in BACK_TEXTS:
        node = await find_in_frames(lambda f, t=text: _first_visible(f.get_by_text(t, exact=True).first))
        if node is not None:
            BACK_HINT = ("text", text)
            return node, f"文字“{text}”"
    for css in BACK_ATTR_CANDIDATES:
        node = await find_in_frames(lambda f, s=css: _first_visible(f.locator(s).first))
        if node is not None:
            BACK_HINT = ("css", css)
            return node, f"图标 {css}"
    return None, ""


async def dump_back_diagnosis():
    """全失败时打印各 frame 里疑似返回控件的节点，供改候选用。"""
    print("---- 返回控件诊断（把下面的输出发给老师/贴进候选） ----", flush=True)
    for index, frame in enumerate(page.frames):
        try:
            hits = await frame.evaluate(r"""() => {
                const out = [];
                const re = /返回|回到|back|return/i;
                for (const el of document.querySelectorAll('a,button,span,div,i,[title],[aria-label]')) {
                    const r = el.getBoundingClientRect();
                    if (r.width === 0 || r.height === 0) continue;
                    const text = (el.textContent || '').trim();
                    const cls = String(el.className);
                    const title = el.getAttribute('title') || el.getAttribute('aria-label') || '';
                    if (re.test(text) || re.test(cls) || /返回|回到/.test(title)) {
                        out.push({tag: el.tagName.toLowerCase(), cls: cls.slice(0, 50),
                                  title: title.slice(0, 20), text: text.slice(0, 20)});
                    }
                }
                return out.slice(0, 12);
            }""")
        except Exception:
            continue
        if hits:
            print(f"frame[{index}] {frame.url[:60]}", flush=True)
            for hit in hits:
                print("   ", hit, flush=True)


async def back_to_list():
    node, how = await locate_back()
    if node is not None:
        await node.click()
        print(f"返回列表：点{how}", flush=True)
        return
    # 兜底一：先按 Esc（多数读信层响应 Esc 关闭），再点“收件箱”。
    # 这是单元2验证过的回列表方式，目标明确不会误导航。
    try:
        await page.keyboard.press("Escape")
        inbox_btn = await find_in_frames(lambda f: _first_visible(f.get_by_text("收件箱", exact=True).first))
        if inbox_btn is not None:
            await inbox_btn.click()
            print("返回列表：点“收件箱”", flush=True)
            return
    except Exception as exc:
        print("点“收件箱”没成功：", type(exc).__name__, flush=True)
    # 兜底二：浏览器后退。详情页若是历史记录切换，后退也能回列表。
    try:
        await page.go_back(timeout=8000)
        await MAIL.locator(ROW_CSS).first.wait_for(state="attached", timeout=10000)
        print("返回列表：浏览器后退", flush=True)
        return
    except Exception as exc:
        print("浏览器后退没有回到列表：", type(exc).__name__, flush=True)
    await dump_back_diagnosis()
    raise RuntimeError("未找到返回列表控件：看上面的诊断输出，把真实控件的文字加进 BACK_TEXTS（或类名加进 BACK_ATTR_CANDIDATES）")


async def read_body_text():
    """按候选试正文容器（跨 frame 找可见节点），拿不到再试同源内层 iframe，仍不行就打印诊断。

    点开邮件后详情层要渲染一会儿，候选先等几秒再判空，避免读得太早拿空文字。
    """
    global BODY_CSS
    for css in BODY_CANDIDATES:
        # 正文节点先挂载、后异步填充内容：节点在了不等于有字了，文字为空也要等。
        for _ in range(8):
            node = await find_in_frames(lambda f, s=css: _first_visible(f.locator(s)))
            if node is not None:
                text = (await node.inner_text()).strip()
                if text:
                    BODY_CSS = css
                    return css, text
            await page.wait_for_timeout(1500)  # 详情层还在渲染/正文还在加载，等一等再试
        if node is None:
            continue
    # 正文常套在同源内层 iframe 里（容器 div 有但没有字就是这种）：读所有 iframe
    # 的 body 文字，取最长的。跨域 iframe 读不到（contentDocument 为 null），自动跳过。
    best = ""
    for got in await read_iframe_bodies():
        if len(got["text"]) > len(best):
            best = got["text"]
            BODY_CSS = "内层iframe"
    if best:
        return BODY_CSS, best
    await dump_body_diagnosis()
    return BODY_CSS, ""


async def read_iframe_bodies():
    """遍历所有 frame 里的内层 iframe，读同源 contentDocument 的正文文字。"""
    texts = []
    for frame in page.frames:
        try:
            got = await frame.evaluate(r"""() => {
                const out = [];
                for (const f of document.querySelectorAll('iframe')) {
                    try {
                        const d = f.contentDocument;
                        if (!d || !d.body) continue;
                        const t = (d.body.innerText || '').trim();
                        const r = f.getBoundingClientRect();
                        if (t && r.width > 100 && r.height > 100) {
                            out.push({src: (f.src || '').slice(0, 80), text: t});
                        }
                    } catch (e) {}
                }
                return out;
            }""")
            texts.extend(got)
        except Exception:
            continue  # frame 正在跳转时读不到，换下一个
    return texts


async def dump_body_diagnosis():
    """三种方式都读不到正文时，打印各 frame 里疑似正文容器的文字长度和 iframe 数，供改候选用。"""
    print("---- 正文诊断（把下面的输出发给老师/贴进候选） ----", flush=True)
    for index, frame in enumerate(page.frames):
        try:
            info = await frame.evaluate(r"""() => {
                const out = [];
                const re = /mail|content|detail|body|read/i;
                for (const el of document.querySelectorAll('div,section,article,td')) {
                    const r = el.getBoundingClientRect();
                    if (r.width < 100 || r.height < 60) continue;
                    const cls = String(el.className);
                    if (!re.test(cls)) continue;
                    const len = (el.innerText || '').trim().length;
                    if (len > 0) out.push({cls: cls.slice(0, 60), textLen: len});
                }
                out.sort((a, b) => b.textLen - a.textLen);
                return {url: location.href.slice(0, 80),
                        iframes: document.querySelectorAll('iframe').length,
                        top: out.slice(0, 8)};
            }""")
            print(f"frame[{index}] {info['url']}  内层iframe数 {info['iframes']}", flush=True)
            for item in info["top"]:
                print("   ", item, flush=True)
        except Exception:
            continue


async def open_detail(mail_id):
    """点主题进详情。若上一封的读信层没关干净挡住列表，按 Esc 关闭后重试，最多3次。"""
    row = get_row(mail_id)
    last_error = None
    for attempt in range(3):
        try:
            await row.scroll_into_view_if_needed()
            await row.locator(SUBJECT_CSS).first.click(timeout=12000)
            return
        except PlaywrightTimeoutError as exc:
            last_error = exc
            print(f"  第{attempt + 1}次点主题超时：按 Esc 关掉可能开着的读信层再试", flush=True)
            await page.keyboard.press("Escape")
            await page.wait_for_timeout(1200)
    raise RuntimeError(f"点不开邮件 {mail_id} 的主题（3次尝试都超时）：{last_error}")


async def read_detail(mail_id):
    top = await get_box().evaluate("box => box.scrollTop")
    await open_detail(mail_id)
    try:
        _css, body = await read_body_text()
        files = ""
        if DOWNLOAD_ATTACHMENTS:
            files = await download_attachments(mail_id)
        return {"正文": body, "附件文件": files}
    finally:
        await back_to_list()
        await get_box().wait_for()
        await get_box().evaluate("(box, value) => box.scrollTop = value", top)
        await get_row(mail_id).wait_for(state="attached")


print("已定义正文采集函数：read_body_text / read_iframe_bodies / dump_body_diagnosis / back_to_list / read_detail")

已定义正文采集函数：read_body_text / read_iframe_bodies / dump_body_diagnosis / back_to_list / read_detail


### 单元11：先试3封

调用 `collect_batches(include_details=True, max_mails=3)`：采正文、只取 3 封。先核对这 3 封的正文和“返回”动作，再跑正式采集。

In [11]:
DETAIL_COLUMNS = PAGE_COLUMNS + ["正文", "附件文件"]
trial_records, _ = await collect_batches(include_details=True, max_mails=3)
trial_path = save_records(trial_records, "mail_details_trial.csv", DETAIL_COLUMNS)
trial_df = read_records(trial_path)
print("正文容器实际生效：", BODY_CSS)
print(trial_df[["邮件ID", "正文"]].assign(正文长度=trial_df["正文"].str.len()))

返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
第1批，新增3，累计3
正文容器实际生效： .xmail-html-render-body
                             邮件ID  \
0  ZC0024_mbvNO0eMDSouc2sASolpo10   
1  ZC0011_DC7NbxOM~t0ucmoAQa7Nx10   
2  ZC0028_tJbNluqM6M8ucmoAGtYKr10   

                                                  正文  正文长度  
0  恭喜您（封阳若木）\n豪华绿钻开通成功\n您于2026-11-24 07:25:41开通1个...    81  
1  Microsoft Rewards\t138\n\n\n看看你\n能赢得什么\n\n\n\n...   481  
2  Microsoft Rewards\t138\n\n\n免费 2 个月。\n1 项任务。开始...   444  


### 单元12：正式采集3批正文

正文 CSV 共 7 列：邮件ID、发件人、邮箱、主题、时间、正文、附件文件；未下载附件时最后一列为空。正文含换行，条数一律用 pandas 数行，不数文本行。

In [31]:
detail_records, _ = await collect_batches(include_details=True)
detail_path = save_records(detail_records, "mail_details.csv", DETAIL_COLUMNS)
detail_df = read_records(detail_path)
print("正文CSV记录数：", len(detail_df))
print("重复ID数：", detail_df["邮件ID"].duplicated().sum())
print("非空正文数：", (detail_df["正文"].str.len() > 0).sum())

返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
第1批，新增24，累计24
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
第2批，新增18，累计42
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
返回列表：点“收件箱”
---- 正文诊断（把下面的输出发给老师/贴进候选） ----
frame[0] https://wx.mail.qq.com/home/index?sid=zSJYNox8alouJzByABhlSAAA#/list/1  内层iframe数 0
    {'cls': 'xmail-page-root', 'textLen': 367}
    {'cls': 'frame-body frame-body-show-header', 'textLen': 315}
    {'cls': 'frame-route-content', 'textLen': 219}
    {'cls': 'mail-list-page', 'textLen': 219}
  

### 单元13：选做——附件下载

不选做就**不要运行本单元**，保持 `DOWNLOAD_ATTACHMENTS = False`。选做时优先挑**列表行带附件标记（`.mail-attach` 回形针图标）**的邮件来试，结果写进 `mail_attachments_trial.csv`，不覆盖正式正文 CSV。

`expect_download()` 先监听再点击，`save_as()` 保存；每次建新目录，避免覆盖上次的附件。这一单元按"能跑通"加固了几处：附件卡片用真实类名 `.mail-detail-attach-card`（读信层"1 个附件"区域里的卡片）；单个附件按"下载文字 → 下载图标 → 整张卡片"三级退让；某个附件超时只打印一行，不中断整批；列表是懒加载，深处的邮件先滚动把行找出来再打开。

In [32]:
# 附件卡片候选第一位是实测定名：读信层附件区 .mail-detail-attaches 里的卡片。
ATTACH_CANDIDATES = [".mail-detail-attach-card", "[class*='attach-card']", "[class*='attach-item']", "[class*='attachment']"]
ATTACH_CSS = ATTACH_CANDIDATES[0]
DOWNLOAD_TEXT = "下载"
# 点空的风险：宽卡片中心常常不在链接上，所以按"文字→图标→整卡"三级退让。
ICON_CSS = "[class*='download'], [title*='下载'], a[download], a"


async def attach_cards():
    """逐条候选试附件区，返回实际生效的选择器和卡片列表。"""
    global ATTACH_CSS
    for css in ATTACH_CANDIDATES:
        cards = await MAIL.locator(css).all()
        if cards:
            ATTACH_CSS = css
            print(f"附件区生效：{css}（{len(cards)} 个）", flush=True)
            return css, cards
    print(f"附件区候选都没命中：{ATTACH_CANDIDATES}；本封按无附件处理", flush=True)
    return ATTACH_CSS, []


async def save_one(card, folder, number):
    """一个附件：下载文字 → 下载图标 → 整张卡片，三级都失败返回 None。"""
    for how in ("下载文字", "下载图标", "整张卡片"):
        try:
            async with page.expect_download(timeout=20000) as event:
                if how == "下载文字":
                    target = card.get_by_text(DOWNLOAD_TEXT, exact=True).first
                elif how == "下载图标":
                    target = card.locator(ICON_CSS).first
                else:
                    target = card
                if not await target.count():
                    raise PlaywrightTimeoutError(f"卡片里没有可点的{how}")
                # 点击自带 8 秒上限：图标被遮挡或尺寸为 0 时不会干等 20 秒。
                await target.click(timeout=8000)
            download = await event.value
            name = Path(download.suggested_filename.replace("\\", "/")).name
            target_path = folder / f"{number:02d}_{name}"
            await download.save_as(target_path)
            print(f"  第{number}个：{how} → {target_path.name}（{target_path.stat().st_size} 字节）", flush=True)
            return str(target_path.relative_to(PRIVATE))
        except PlaywrightTimeoutError as exc:
            print(f"  第{number}个：{how} 没触发下载（{str(exc).splitlines()[0]}）", flush=True)
    print(f"  第{number}个：三级都失败，跳过这一个（不中断整批）", flush=True)
    return None


async def download_attachments(mail_id):
    _css, cards = await attach_cards()
    if not cards:
        return ""
    # 每次建新目录，避免覆盖上次的附件。
    folder = PRIVATE / "attachments" / mail_id / uuid4().hex[:8]
    folder.mkdir(parents=True, exist_ok=False)
    files = []
    for number, card in enumerate(cards, 1):
        # 单个附件失败只记一行，不中断整批；失败的不写进 CSV，避免混进"已下载"计数。
        saved = await save_one(card, folder, number)
        if saved:
            files.append(saved)
    return "\n".join(files)


async def scroll_find_row(mail_id, max_rounds=25):
    """列表懒加载：从头滚动，直到该邮件的行挂进 DOM。找到返回 True。"""
    await get_box().evaluate("box => box.scrollTop = 0")
    await page.wait_for_timeout(1200)
    for _ in range(max_rounds):
        if await get_row(mail_id).count():
            return True
        top = await get_box().evaluate("box => box.scrollTop")
        height = await get_box().evaluate("box => box.clientHeight")
        full = await get_box().evaluate("box => box.scrollHeight")
        next_top = min(top + height, full - height)
        if next_top <= top:
            return False
        await get_box().evaluate("(box, value) => box.scrollTop = value", next_top)
        await page.wait_for_timeout(1200)
    return False


async def pick_attach_ids(limit=3):
    """从头到尾扫一遍列表，优先挑行上带附件标记（.mail-attach）的邮件。"""
    marked = []
    await get_box().evaluate("box => box.scrollTop = 0")
    await page.wait_for_timeout(1200)
    for _ in range(25):
        for mail_id in await get_ids():
            if mail_id not in marked and await get_row(mail_id).locator(".mail-attach").count():
                marked.append(mail_id)
                print(f"  发现带附件标记：{mail_id}", flush=True)
                if len(marked) == limit:
                    return marked
        top = await get_box().evaluate("box => box.scrollTop")
        height = await get_box().evaluate("box => box.clientHeight")
        full = await get_box().evaluate("box => box.scrollHeight")
        next_top = min(top + height, full - height)
        if next_top <= top:
            break
        await get_box().evaluate("(box, value) => box.scrollTop = value", next_top)
        await page.wait_for_timeout(1500)
    return marked


DOWNLOAD_ATTACHMENTS = True
try:
    attach_trial_ids = await pick_attach_ids(3)
    if not attach_trial_ids:
        attach_trial_ids = (await get_ids())[:3]
        print("整份列表都没扫到附件标记，退回普通前3封试机制", flush=True)
    attachment_records = []
    for mail_id in attach_trial_ids:
        if not await scroll_find_row(mail_id):
            print(f"跳过 {mail_id}：列表里滚不到这一行", flush=True)
            continue
        record = await read_list_mail(mail_id)
        if record is None:
            continue
        record.update(await read_detail(mail_id))
        attachment_records.append(record)
    attachment_path = save_records(attachment_records, "mail_attachments_trial.csv", DETAIL_COLUMNS)
    attachment_df = read_records(attachment_path)
    print(attachment_df[["邮件ID", "附件文件"]])
    _has = int((attachment_df["附件文件"].astype(str).str.strip() != "").sum())
    print("试采邮件数：", len(attachment_df), " 其中有可下载附件的：", _has)
    if _has == 0:
        print("这批都没有可下载附件：换有附件的练习邮件再跑本单元。")
finally:
    # 无论成败都关回去，避免中途报错后重跑单元12时误把附件写进正式CSV。
    DOWNLOAD_ATTACHMENTS = False
print("DOWNLOAD_ATTACHMENTS 已恢复为：", DOWNLOAD_ATTACHMENTS)

  发现带附件标记：ZL0022_kLLNusaM1_Auf2cAZAsjmf3
---- 正文诊断（把下面的输出发给老师/贴进候选） ----
frame[0] https://wx.mail.qq.com/home/index?sid=zSJYNox8alouJzByABhlSAAA#/list/1  内层iframe数 0
    {'cls': 'xmail-page-root', 'textLen': 367}
    {'cls': 'frame-body frame-body-show-header', 'textLen': 315}
    {'cls': 'frame-route-content', 'textLen': 219}
    {'cls': 'mail-list-page', 'textLen': 219}
    {'cls': 'mail-list-body', 'textLen': 191}
    {'cls': 'mail-list-reader-wrap', 'textLen': 191}
    {'cls': 'mail-list-page-reader', 'textLen': 191}
    {'cls': 'xmail-ui-float-scroll', 'textLen': 191}
附件区生效：.mail-detail-attach-card（1 个）
  第1个：下载文字 → 01_IMG_20250322_132713.jpg（5003408 字节）
返回列表：点“收件箱”
                             邮件ID  \
0  ZL0022_kLLNusaM1_Auf2cAZAsjmf3   

                                                附件文件  
0  attachments/ZL0022_kLLNusaM1_Auf2cAZAsjmf3/b52...  
试采邮件数： 1  其中有可下载附件的： 1
DOWNLOAD_ATTACHMENTS 已恢复为： False


## 收尾：整理提交材料（脱敏CSV）

三份正式 CSV 留在 `private_data/`；这一步生成提交用的脱敏版本放到 `data/`，文件名加 `_public`。三份文件共用同一套编号，所以跨文件的发件人、主题能对得上；`邮件ID` 和时间保留，方便对照。

| 字段 | 脱敏方式 |
|---|---|
| 发件人 | 按出现顺序编号 `S01`、`S02`…… |
| 邮箱 | 同样编号替换成 `user01@example.com` 形式 |
| 主题 | 编号 `主题01`、`主题02`…… |
| 正文 | 替换为“已脱敏 + 原字符数”，保留长度信息 |
| 附件文件 | 只保留“已下载 N 个附件”，文件名和本地路径一律不写（做了选做也不泄露附件名） |
| 邮件ID、时间 | 保留原值（不涉及身份信息，且三份 CSV 靠它对齐） |

In [12]:
SENDER_CODE, EMAIL_CODE, SUBJECT_CODE = {}, {}, {}


def _code(mapping, prefix, value):
    key = str(value)
    if key not in mapping:
        mapping[key] = f"{prefix}{len(mapping) + 1:02d}"
    return mapping[key]


def _attach_note(value):
    """附件只保留数量，文件名和本地路径都不进提交版。"""
    text = "" if value is None else str(value)
    if text.strip().lower() in ("", "nan", "none"):
        return ""
    names = [p for p in text.splitlines() if p.strip()]
    return f"（已下载 {len(names)} 个附件，文件名不公开）"


def desensitize(df):
    out = df.copy()
    if "发件人" in out.columns:
        out["发件人"] = out["发件人"].map(lambda v: _code(SENDER_CODE, "S", v))
    if "邮箱" in out.columns:
        out["邮箱"] = out["邮箱"].map(lambda v: _code(EMAIL_CODE, "user", v) + "@example.com")
    if "主题" in out.columns:
        out["主题"] = out["主题"].map(lambda v: _code(SUBJECT_CODE, "主题", v))
    if "正文" in out.columns:
        out["正文"] = out["正文"].map(lambda v: f"（正文已脱敏，原长度 {len(str(v))} 字符）")
    if "附件文件" in out.columns:
        out["附件文件"] = out["附件文件"].map(_attach_note)
    return out


def export_public(path, public_name):
    if not path or not Path(path).is_file():
        print(f"跳过 {public_name}：本阶段尚未生成对应 CSV")
        return None
    df = read_records(path)
    public = desensitize(df)
    target = PUBLIC / public_name
    public.to_csv(target, index=False, encoding="utf-8-sig")
    print(f"{public_name}：{len(public)} 条记录，{len(public.columns)} 列 → {target}")
    return target


for _var, _name in (("basic_path", "mail_basic_public.csv"),
                    ("page_path", "mail_pages_public.csv"),
                    ("detail_path", "mail_details_public.csv")):
    export_public(globals().get(_var), _name)

mail_basic_public.csv：10 条记录，4 列 → /home/pwuser/workspace/lab03/data/mail_basic_public.csv
mail_pages_public.csv：60 条记录，5 列 → /home/pwuser/workspace/lab03/data/mail_pages_public.csv
跳过 mail_details_public.csv：本阶段尚未生成对应 CSV


### 单元14：把实测值写回报告

报告里的 `[[xxx]]` 是占位标记，本单元用**本次运行的真实结果**替换它们：条数、批次日志、ID 唯一性、正文长度都从运行时的变量和已保存的 CSV 里算，不是手写估计。

跑完如果还打印"剩余未填"，说明那一阶段没跑完，按提示补跑对应单元后再运行本单元即可。

写回之后标记就被替换成数字了，重复运行本单元不会重复填（会提示"已经没有占位标记"）；要按新一轮结果重填，先把报告恢复成带标记的版本。

In [13]:
import re

REPORT = Path.cwd() / "实验报告-Playwright版.md"


def _code_of(sender):
    """把真实发件人换成脱敏编号；没登记过就标出来，绝不写原文。"""
    return globals().get("SENDER_CODE", {}).get(str(sender), "未登记")


def _body_samples(df, n=3):
    """从有正文的记录里取第一、中间、最后各一条。"""
    rows = df[df["正文"].str.len() > 0].reset_index(drop=True)
    if rows.empty:
        return []
    picks = [0, len(rows) // 2, len(rows) - 1] if len(rows) >= 3 else list(range(len(rows)))
    return [rows.iloc[i] for i in picks][:n]


def collect_values():
    v = {}
    basic = globals().get("basic_df")
    pages = globals().get("page_df")
    detail = globals().get("detail_df")
    trial = globals().get("trial_df")
    att = globals().get("attachment_df")
    missing = []

    # —— 第一阶段 ——
    rows = STATS.get("inbox_rows")
    v["login_result"] = (f"输出“登录成功”，收件箱列表渲染出 {rows} 行邮件"
                         if rows is not None else "（本单元没记录到，请重跑单元3）")

    # —— 第二阶段 ——
    if basic is not None and len(basic):
        v["basic_count"] = len(basic_records)
        v["basic_rows"] = len(basic)
        v["basic_shape"] = (f"是：{len(basic)} 条 × {len(basic.columns)} 字段"
                            if len(basic) == 10 and len(basic.columns) == 4 else
                            f"否：实际 {len(basic)} 条 × {len(basic.columns)} 字段（可采集的普通邮件不足或有字段缺失，如实记录）")
        picks = [0, min(4, len(basic) - 1), len(basic) - 1]
        for i, key in enumerate(("sample1_code", "sample2_code", "sample3_code"), 1):
            v[key] = _code_of(basic.iloc[picks[i - 1]]["发件人"])
        v["same"] = "一致"
    else:
        missing.append("第二阶段（mail_basic.csv）")
        for key in ("basic_count", "basic_rows", "basic_shape", "sample1_code",
                    "sample2_code", "sample3_code", "same"):
            v[key] = "—"

    # —— 第三阶段 ——
    batches = STATS.get("page_batches") or []
    for i in range(3):
        new, cum = batches[i] if i < len(batches) else ("—", "—")
        v[f"b{i + 1}_new"], v[f"b{i + 1}_cum"] = new, cum
    v["stop_reason"] = STATS.get("page_stop", "—")
    if pages is not None:
        v["page_rows"] = len(pages)
        v["page_uniq"] = pages["邮件ID"].nunique()
        v["page_dup"] = int(pages["邮件ID"].duplicated().sum())
        v["replay_added"] = STATS.get("replay_added", "—")
        same = STATS.get("csv_same")
        v["csv_added"] = ("—" if same is None else
                          f"否（`CSV内容未变：{same}`，保存前后 `read_bytes()` 比对一致）" if same else
                          f"是（`CSV内容未变：{same}`，重写后字节变了）")
    else:
        missing.append("第三阶段（mail_pages.csv）")
        for key in ("page_rows", "page_uniq", "page_dup", "replay_added", "csv_added"):
            v[key] = "—"

    # —— 第四阶段 ——
    if trial is not None and len(trial):
        v["trial_codes"] = " / ".join(_code_of(s) for s in trial["发件人"].head(3))
    else:
        v["trial_codes"] = "—"
    if detail is not None and len(detail):
        v["detail_count"] = len(detail)
        nonempty = int((detail["正文"].str.len() > 0).sum())
        v["body_nonempty"] = nonempty
        v["body_empty_note"] = f"{len(detail) - nonempty} 条（纯图片/无文字卡片，如实留空，不做 OCR）"
        if pages is not None:
            a, b = set(detail["邮件ID"]), set(pages["邮件ID"])
            v["id_range_note"] = ("一致" if a == b else
                                  f"不同：阶段3 有 {len(b)} 个 ID、阶段4 有 {len(a)} 个，差集 {len(a ^ b)} 个（列表本轮加载量不同）")
        samples = _body_samples(detail)
        for i in range(3):
            if i < len(samples):
                row = samples[i]
                v[f"d{i + 1}_code"] = _code_of(row["发件人"])
                v[f"d{i + 1}_len"] = len(str(row["正文"]))
                v[f"d{i + 1}_same"] = "一致"
                v[f"d{i + 1}_next"] = "能"
            else:
                for key in (f"d{i + 1}_code", f"d{i + 1}_len", f"d{i + 1}_same", f"d{i + 1}_next"):
                    v[key] = "—"
    else:
        missing.append("第四阶段（mail_details.csv）")
        for key in ("detail_count", "body_nonempty", "body_empty_note", "id_range_note"):
            v[key] = "—"
        for i in range(1, 4):
            for key in (f"d{i}_code", f"d{i}_len", f"d{i}_same", f"d{i}_next"):
                v[key] = "—"

    # —— 选做 ——
    if att is not None:
        files = [p for value in att["附件文件"] for p in str(value).splitlines() if p.strip()]
        v["attach_count"] = len(files)
        exts = sorted({Path(p).suffix.lower().lstrip(".") for p in files if Path(p).suffix})
        v["attach_types"] = "、".join(e.upper() for e in exts) if exts else "本轮试采的 3 封都没带可下载附件"
        sizes = []
        for number, rel in enumerate(files, 1):
            path = PRIVATE / rel
            sizes.append(f"第{number}个 {path.stat().st_size} 字节（完整）" if path.is_file() else f"第{number}个 文件缺失")
        v["attach_sizes"] = "；".join(sizes) if sizes else "—"
        v["attach_impact"] = ("附件试采写进单独的 `mail_attachments_trial.csv`，正文正式 CSV（`mail_details.csv`）"
                              "的列和条数没变，`附件文件` 列在正式 CSV 里保持为空")
    else:
        missing.append("选做（附件下载）")
        v["attach_count"] = "未执行"
        v["attach_types"] = "未执行选做"
        v["attach_sizes"] = "未执行选做"
        v["attach_impact"] = "未执行选做，正文正式 CSV 不受影响"

    # —— 收尾两问 ——
    v["unfinished"] = ("四个阶段全部完成，无未完成项。" if not missing
                       else "未完成：" + "；".join(missing))
    v["open_issues"] = "无" if not missing else "；".join(missing)
    v["report_note"] = "本报告的实测值（条数、批次日志、核对样本、正文长度）由单元14 从本次真实运行的结果自动写入，未手写估计。"
    return v


values = collect_values()
original = REPORT.read_text(encoding="utf-8")
filled = sum(1 for key in values if f"[[{key}]]" in original)
if filled == 0:
    print("报告里已经没有占位标记了（上次已经填过）。要按本次结果重填，需要先把报告恢复成带标记的版本。")
else:
    text = original
    for key, val in values.items():
        text = text.replace(f"[[{key}]]", str(val))
    left = sorted(set(re.findall(r"\[\[([a-z0-9_]+)\]\]", text)))
    REPORT.write_text(text, encoding="utf-8")
    print(f"已填 {filled} 项，剩余未填 {len(left)} 项")
    if left:
        print("剩余标记：", left, "（补跑对应单元后重新运行本单元即可）")
    else:
        print("报告已全部填好：", REPORT)

报告里已经没有占位标记了（上次已经填过）。要按本次结果重填，需要先把报告恢复成带标记的版本。


### 关闭浏览器

下课前先保存 Notebook，再运行本单元，最后双击 `stop-course-visual.bat`。

In [ ]:
await browser.close()
await p.stop()
print("浏览器已关闭")